## 1.
Используемые функции:

*Biggs Exp02 Function:*
$$ f(x) = \sum_{i=1}^{10} \left( e^{-t_i x_1} - 5 e^{-t_i x_2} - \left(e^{-t_i} - 5 e^{-10 t_i}\right) \right)^2 $$
*Egg Crate Function:*
$$ f(x,y)=x^2 + y^2 + 25(sin^2(x) + sin^2(y)) $$
*Bird Function:*
$$ f(x, y) = sin(x)e^{(1-cos(y))^2}+cos(y)e^{(1-sin(x))^2}+(x-y)^2 $$

Дилемма птицы и яйца решена, сначала было яйцо.

In [5]:
from benchmarkfcns import biggsexp02, eggcrate, bird
from scipy.optimize import minimize
import numpy as np

class Tester:
    def __init__(self, bench, points, minima):
        self.bench_name = bench.__name__
        self.bench = lambda x: Tester.wrapper(bench, x)
        self.points = points
        self.minima = minima
        self.methods = "nelder_mead", "gradient_descent", "bfgs"
        self.trajectory = []

    def wrapper(fun, point):
        """
        Simple wrapper to handle the disrepancy in benchmarkfcns and minimize's input requirements:
        benchmarkfcns requires its input to be in a [[x, y]] format, while minimize requires [x, y].
        """
        return fun([point])

    def callback(self, x):
        """To keep track of the iteration trajectory."""
        self.trajectory.append(x.copy())
    
    def nelder_mead(self):
        results = []
        for point in self.points:
            results.append((point, minimize(self.bench, point, method="Nelder-Mead", callback=self.callback)))
        return results
    
    def gradient_descent(self):
        results = []
        for point in self.points:
            results.append((point, minimize(self.bench, point, method="CG", callback=self.callback)))
        return results

    def bfgs(self):
        results = []
        for point in self.points:
            results.append((point, minimize(self.bench, point, method='bfgs', callback=self.callback)))
        return results

testers = [
    Tester(
        biggsexp02,
        [[1, 10], [2, 9], [20, 20], [0, 0]],
        [[1, 10]],
    ),
    Tester(
        eggcrate,
        [[0, 0], [1, 1], [-5, -3], [-0.5, 0.2]],
        [[0, 0]],
    ),
    Tester(
        bird,
        [[5, 3], [-1.6, -3], [0, 0], [-2*np.pi, 2*np.pi]],
        [[4.70104, 3.15294], [-1.58214, -3.13024]],
    ),
]

## 2.

In [ ]:
import csv

for tester in testers:
    minima = tester.minima
    for method_name in tester.methods:
        method = getattr(tester, method_name)
        results = method()
        
        with open(f"{tester.bench_name}-{method_name}.csv", "w", newline='') as file:
            writer = csv.writer(file)
            for pair in results:
                point = pair[0]
                res_obj = pair[1]
                writer.writerow([
                    point,
                    res_obj.get("nit"),
                    res_obj.get("nfev"),
                    res_obj.get("njev"),
                    res_obj.get("nhev")] +
                    [np.linalg.norm(res_obj.get("x") - np.array(x)) for x in minima],
                    )

## 3.

Анализ в отдельном `.pdf` файле.